<a href="https://colab.research.google.com/github/vinayprabhu/AMPversity/blob/main/code/paperclip_amp_crawler_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paperclip-only AMP Corpus Builder

**Goal:** build a high-recall, auditable corpus of antimicrobial peptide sequences from **peer-reviewed PMC papers available through Paperclip**, including supplementary files.

Outputs:
- `amp_evidence.tsv` — evidence-granular rows
- `amp_master_YYYYMMDD.tsv` — exactly `[sequence, wetlab, source, notes]`
- `paper_inventory.tsv` — candidate papers and provenance
- `supplement_inventory.tsv` — supplementary files examined
- checkpoint files so the crawl can resume after Colab disconnects

**Hard source rule:** scientific data are retrieved **only through Paperclip**. Local Python is used only to parse, normalize, deduplicate, and save what Paperclip returned.

Paperclip docs used to build this notebook: v0.7.36.

## 0. Colab setup

Your Colab secret is expected to be named **`PAPAERCLIP_REAGENT`** exactly as supplied.

The notebook copies that secret into Paperclip's documented `PAPERCLIP_API_KEY` environment variable. It never prints the key.

In [2]:
import sys
import importlib
importlib.reload(sys)
#######################
from google.colab import drive
drive.flush_and_unmount()
import os
drive.mount('/gdrive', force_remount=True)
# Enter your own proj_dir here
proj_dir='/gdrive/My Drive/AMP_FELLOWSHIP/code/'
os.chdir(proj_dir)
os.listdir()

Mounted at /gdrive


['apex_mic_inference.py',
 'apex-pathogen',
 'apex',
 '__pycache__',
 'data_paper_apex1',
 'df_vep_raw_paper.csv',
 'df_ara_raw_paper.csv',
 'data',
 'APEX_models_comparison.ipynb',
 'AMPversity.ipynb',
 'paperclip_amp_crawler_colab.ipynb']

In [3]:
# Install Paperclip CLI using Paperclip's own installer.
!curl -fsSL https://paperclip.gxl.ai/install.sh | bash

import os, sys, subprocess, json, re, time, csv, hashlib, shutil, tempfile
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

# Add typical installer locations to PATH.
os.environ["PATH"] = os.path.expanduser("~/.paperclip/bin") + ":" + os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

from google.colab import userdata
api_key = userdata.get("PAPAERCLIP_REAGENT")
assert api_key and api_key.startswith("gxl_"), "Missing/invalid Colab secret: PAPAERCLIP_REAGENT"
os.environ["PAPERCLIP_API_KEY"] = api_key

# Verify Paperclip without revealing the credential.
p = subprocess.run(["paperclip", "config"], capture_output=True, text=True)
print(p.stdout or p.stderr)
assert p.returncode == 0, "Paperclip CLI setup failed."


Paperclip installer
─────────────────────────────────────

✓ Python 3.12 detected
✓ Downloading Paperclip from https://paperclip.gxl.ai...
✓ Installing to /root/.paperclip...
✓ Dependencies satisfied (requests, click, pyyaml)
✓ Paperclip installed successfully

  Binary:  /root/.local/bin/paperclip
  Library: /root/.paperclip/lib/
  Python SDK: import gxl_paperclip — PaperclipClient for scripts and notebooks
               Docs: https://paperclip.gxl.ai


  export PATH="$HOME/.local/bin:$PATH"

  Add to /root/.bashrc? [Y/n] bash: line 142: /dev/tty: No such device or address
✓ Added to /root/.bashrc
─────────────────────────────────────
bash: line 169: /dev/tty: No such device or address

  Paperclip
  Server:  https://paperclip.gxl.ai
           (default)
  Auth:    ✓ API key (env)
  Config:  /root/.paperclip
  Health:  ✓ server reachable
  Sources: PubMed Central, bioRxiv, medRxiv, arXiv




## 1. Configuration

Start with `MODE="pilot"` to validate the pipeline. Then change to `MODE="full"`.

`full` uses corpus-wide Paperclip grep plus a broad search union. This is intentionally recall-biased.

In [4]:
MODE = "full"          # "pilot" or "full"
OUT = Path("/content/amp_paperclip")
OUT.mkdir(parents=True, exist_ok=True)

MIN_LEN = 8
MAX_LEN = 50
CANONICAL_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Peer-reviewed source restriction.
SOURCE = "pmc"

# Pilot limits only. Full mode has no paper-count cap after discovery.
PILOT_PAPERS = 50

# Conservative exact-sequence regex. Allows only the 20 canonical amino acids.
SEQ_RE = re.compile(rf"(?<![A-Z])([ACDEFGHIKLMNPQRSTVWY]{{{MIN_LEN},{MAX_LEN}}})(?![A-Z])")

# High-recall discovery vocabulary.
DISCOVERY_QUERIES = [
    "antimicrobial peptide",
    "antibacterial peptide",
    "bactericidal peptide",
    "host defense peptide antimicrobial",
    "cationic antimicrobial peptide",
    "membrane active antimicrobial peptide",
    "peptide minimum inhibitory concentration",
    "peptide MIC antibacterial",
    "peptide MIC50 antibacterial",
    "peptide MIC90 antibacterial",
    "peptide hemolysis MIC",
    "peptide HC50 antimicrobial",
    "antimicrobial peptide sequence",
    "antibacterial peptide sequence",
    "synthetic antimicrobial peptide",
    "designed antimicrobial peptide",
    "novel antimicrobial peptide",
    "AMP antimicrobial peptide bacteria",
    "broad spectrum antimicrobial peptide",
    "anti-biofilm peptide antibacterial",
    "bacteriocin peptide antimicrobial",
    "defensin antimicrobial peptide",
    "cathelicidin antimicrobial peptide",
    "temporin antimicrobial peptide",
    "magainin antimicrobial peptide",
    "cecropin antimicrobial peptide",
]

# Corpus-wide grep terms catch papers missed by ranked search.
GREP_DISCOVERY_PATTERNS = [
    r"antimicrobial peptide",
    r"antibacterial peptide",
    r"minimum inhibitory concentration",
    r"\bMIC\b",
    r"bactericidal peptide",
    r"host defense peptide",
]

# Wet-lab evidence vocabulary. We do NOT equate mere mention with validation.
WETLAB_TERMS = [
    "minimum inhibitory concentration", " MIC ", "MIC50", "MIC90",
    "broth microdilution", "agar dilution", "time-kill", "time kill",
    "colony forming", "CFU", "bactericidal", "MBC",
    "hemolysis", "haemolysis", "HC50", "cytotoxicity",
    "murine", "mouse model", "mice", "in vivo", "infection model"
]

print("Output:", OUT)
print("Mode:", MODE)

Output: /content/amp_paperclip
Mode: full


## 2. Paperclip helper functions

All corpus reads below call the `paperclip` executable. There are **no PubMed, Crossref, Springer, Nature, Google, or other literature HTTP calls** in the crawler.

In [5]:
def pc(args, timeout=300, binary=False, check=True):
    """Run Paperclip CLI. Returns stdout text or bytes."""
    cmd = ["paperclip"] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=not binary, timeout=timeout)
    if check and r.returncode != 0:
        err = r.stderr.decode(errors="replace") if binary and isinstance(r.stderr, bytes) else r.stderr
        raise RuntimeError(f"Paperclip failed ({r.returncode}): {' '.join(cmd)}\n{err}")
    return r.stdout

def pc_json(args, timeout=300):
    out = pc(list(args) + ["--json"], timeout=timeout)
    try:
        return json.loads(out)
    except Exception:
        return out

def extract_pmc_ids(obj):
    """Robustly recover PMC IDs from Paperclip JSON or formatted output."""
    text = obj if isinstance(obj, str) else json.dumps(obj)
    return sorted(set(re.findall(r"\bPMC\d+\b", text)))

def get_meta(pmc):
    out = pc(["cat", f"/papers/{pmc}/meta.json"], timeout=120)
    # Paperclip meta.json should be JSON. Strip accidental CLI framing if needed.
    try:
        return json.loads(out)
    except Exception:
        m = re.search(r"\{.*\}", out, flags=re.S)
        return json.loads(m.group(0)) if m else {"id": pmc, "raw_meta": out}

def list_dir(path):
    out = pc(["ls", path], timeout=120, check=False)
    lines = [x.strip() for x in out.splitlines() if x.strip()]
    # Remove obvious comments/headings if any.
    return [x for x in lines if not x.startswith("#")]

def grep_file(pattern, path, context=2, ignore_case=False, only_matching=False):
    args = ["grep"]
    if ignore_case: args.append("-i")
    if context: args += ["-C", str(context)]
    if only_matching: args.append("-o")
    args += [pattern, path]
    return pc(args, timeout=180, check=False)

def paperclip_binary(path):
    """Read a binary/text VFS file through Paperclip cat only."""
    return pc(["cat", path], timeout=300, binary=True, check=False)

print("Helpers ready.")

Helpers ready.


## 3. High-recall paper discovery

Two independent Paperclip channels are unioned:

1. **Hybrid full-corpus searches**, max 1000 per query.
2. **Corpus-wide regex grep** over `/papers/`, which uses Paperclip's trigram corpus index.

Only `PMC...` IDs are retained, giving us peer-reviewed PMC papers rather than preprints.

In [6]:
paper_ids = set()
search_log = []

for i, q in enumerate(DISCOVERY_QUERIES, 1):
    print(f"[search {i}/{len(DISCOVERY_QUERIES)}] {q}")
    try:
        raw = pc_json(["search", q, "--source", SOURCE, "--all", "-n", "1000"], timeout=300)
        ids = extract_pmc_ids(raw)
        paper_ids.update(ids)
        search_log.append({"method":"search","query":q,"n_ids":len(ids),"status":"ok"})
        print("  +", len(ids), "=> union", len(paper_ids))
    except Exception as e:
        search_log.append({"method":"search","query":q,"n_ids":0,"status":repr(e)})
        print("  ERROR:", e)
    if MODE == "pilot" and len(paper_ids) >= PILOT_PAPERS:
        break

if MODE == "full":
    for i, pat in enumerate(GREP_DISCOVERY_PATTERNS, 1):
        print(f"[grep {i}/{len(GREP_DISCOVERY_PATTERNS)}] {pat}")
        # -l lists matching files only. We recover PMC IDs from paths.
        raw = pc(["grep", "-i", "-l", pat, "/papers/"], timeout=600, check=False)
        ids = extract_pmc_ids(raw)
        paper_ids.update(ids)
        search_log.append({"method":"grep","query":pat,"n_ids":len(ids),"status":"ok"})
        print("  +", len(ids), "=> union", len(paper_ids))

paper_ids = sorted(paper_ids)
if MODE == "pilot":
    paper_ids = paper_ids[:PILOT_PAPERS]

pd.DataFrame(search_log).to_csv(OUT/"discovery_log.tsv", sep="\t", index=False)
(OUT/"paper_ids.txt").write_text("\n".join(paper_ids))
print("\nCandidate PMC papers:", len(paper_ids))

[search 1/26] antimicrobial peptide
  + 500 => union 500
[search 2/26] antibacterial peptide
  + 483 => union 775
[search 3/26] bactericidal peptide
  + 487 => union 1068
[search 4/26] host defense peptide antimicrobial
  + 500 => union 1305
[search 5/26] cationic antimicrobial peptide
  + 488 => union 1477
[search 6/26] membrane active antimicrobial peptide
  + 484 => union 1695
[search 7/26] peptide minimum inhibitory concentration
  + 453 => union 1965
[search 8/26] peptide MIC antibacterial
  + 476 => union 2068
[search 9/26] peptide MIC50 antibacterial
  + 479 => union 2100
[search 10/26] peptide MIC90 antibacterial
  + 479 => union 2109
[search 11/26] peptide hemolysis MIC
  + 484 => union 2356
[search 12/26] peptide HC50 antimicrobial
  + 491 => union 2361
[search 13/26] antimicrobial peptide sequence
  + 489 => union 2471
[search 14/26] antibacterial peptide sequence
  + 476 => union 2502
[search 15/26] synthetic antimicrobial peptide
  + 488 => union 2599
[search 16/26] design

## 4. Build paper inventory and remove obvious reviews from **evidence labeling**

Reviews remain useful discovery maps, but their sequence mentions should not by themselves create `wetlab=yes` evidence.

This notebook keeps them in `paper_inventory.tsv` and marks likely primary vs review from Paperclip metadata.

In [ ]:
inventory_path = OUT/"paper_inventory.tsv"
done = set()
rows = []

if inventory_path.exists():
    old = pd.read_csv(inventory_path, sep="\t", dtype=str).fillna("")
    rows = old.to_dict("records")
    done = set(old["paper_id"])
    print("Resuming inventory:", len(done), "already done")

for j, pmc in enumerate(paper_ids, 1):
    if pmc in done:
        continue
    try:
        m = get_meta(pmc)
        article_type = str(m.get("article_type", m.get("type","")) or "")
        title = str(m.get("title","") or "")
        doi = str(m.get("doi","") or "")
        journal = str(m.get("journal", m.get("journal_title","")) or "")
        date = str(m.get("date", m.get("pub_date","")) or "")
        is_review = "review" in article_type.lower() or title.lower().startswith("review")
        rows.append({
            "paper_id":pmc, "doi":doi, "title":title, "journal":journal,
            "date":date, "article_type":article_type, "is_review":is_review
        })
    except Exception as e:
        rows.append({"paper_id":pmc,"doi":"","title":"","journal":"","date":"",
                     "article_type":"","is_review":"","error":repr(e)})
    if j % 20 == 0:
        pd.DataFrame(rows).to_csv(inventory_path, sep="\t", index=False)
        print(j, "/", len(paper_ids))

inventory = pd.DataFrame(rows)
inventory.to_csv(inventory_path, sep="\t", index=False)
print("Inventory rows:", len(inventory))
display(inventory.head())

20 / 4352
40 / 4352
60 / 4352
80 / 4352
100 / 4352
120 / 4352
140 / 4352
160 / 4352
180 / 4352
200 / 4352
220 / 4352
240 / 4352
260 / 4352
280 / 4352
300 / 4352
320 / 4352
340 / 4352
360 / 4352
380 / 4352
400 / 4352
420 / 4352
440 / 4352
460 / 4352
480 / 4352
500 / 4352
520 / 4352
540 / 4352
560 / 4352
580 / 4352
600 / 4352
620 / 4352
640 / 4352
660 / 4352
680 / 4352
700 / 4352
720 / 4352
740 / 4352
760 / 4352
780 / 4352
800 / 4352
820 / 4352
840 / 4352
860 / 4352
880 / 4352
900 / 4352
920 / 4352
940 / 4352
960 / 4352
980 / 4352
1000 / 4352
1020 / 4352
1040 / 4352
1060 / 4352
1080 / 4352
1100 / 4352
1120 / 4352
1140 / 4352
1160 / 4352
1180 / 4352
1200 / 4352
1220 / 4352
1240 / 4352
1260 / 4352
1280 / 4352
1300 / 4352
1320 / 4352
1340 / 4352
1360 / 4352
1380 / 4352
1400 / 4352
1420 / 4352
1440 / 4352
1460 / 4352
1480 / 4352
1500 / 4352
1520 / 4352
1540 / 4352
1560 / 4352
1580 / 4352
1600 / 4352
1620 / 4352
1640 / 4352
1660 / 4352
1680 / 4352
1700 / 4352
1720 / 4352
1740 / 4352
1760 / 43

## 5. Extract explicit peptide sequences from paper bodies

We ask Paperclip's regex engine for canonical amino-acid strings of length 8–50, then retain nearby line context for provenance.

This is deliberately conservative:
- no peptide name → sequence inference;
- no guessed residues;
- no automatic PTM reconstruction;
- no proteins longer than 50 aa;
- obvious English-word false positives are filtered later using context/evidence.

In [ ]:
body_out = OUT/"body_sequence_candidates.tsv"
existing = pd.DataFrame()
done_body = set()
if body_out.exists():
    existing = pd.read_csv(body_out, sep="\t", dtype=str).fillna("")
    done_body = set(existing["paper_id"].unique())
    print("Resuming body extraction:", len(done_body), "papers done")

body_rows = existing.to_dict("records") if len(existing) else []

# Basic-regex syntax accepted by grep implementations: escaped interval braces.
seq_pattern = rf"[ACDEFGHIKLMNPQRSTVWY]\{{{MIN_LEN},{MAX_LEN}\}}"

for i, pmc in enumerate(paper_ids, 1):
    if pmc in done_body:
        continue
    path = f"/papers/{pmc}/content.lines"
    # Get sequence-containing lines and context directly from Paperclip.
    out = grep_file(seq_pattern, path, context=3, ignore_case=False)
    # Extract every canonical sequence token from the returned context.
    for block in re.split(r"\n--\n", out):
        seqs = sorted(set(SEQ_RE.findall(block)))
        for seq in seqs:
            body_rows.append({
                "paper_id": pmc,
                "sequence": seq,
                "origin": "paper_body",
                "paperclip_path": path,
                "context": " ".join(block.split())[:5000]
            })
    if i % 25 == 0:
        pd.DataFrame(body_rows).drop_duplicates(
            ["paper_id","sequence","origin","paperclip_path"]
        ).to_csv(body_out, sep="\t", index=False)
        print(i, "/", len(paper_ids), "candidate rows", len(body_rows))

body = pd.DataFrame(body_rows)
if len(body):
    body = body.drop_duplicates(["paper_id","sequence","origin","paperclip_path"])
body.to_csv(body_out, sep="\t", index=False)
print("Body sequence candidates:", len(body))

## 6. Enumerate and download supplementary files through Paperclip

This is the critical path for papers whose peptide tables live in supplementary CSV/XLSX/TSV/TXT files.

The notebook never follows publisher URLs. It enumerates `/papers/<PMC>/supplements/` and obtains bytes with `paperclip cat`.

In [ ]:
SUPP_DIR = OUT/"supplements"
SUPP_DIR.mkdir(exist_ok=True)
supp_inv_path = OUT/"supplement_inventory.tsv"

supp_rows = []
done_supp = set()
if supp_inv_path.exists():
    old = pd.read_csv(supp_inv_path, sep="\t", dtype=str).fillna("")
    supp_rows = old.to_dict("records")
    done_supp = set(old["paper_id"].unique())
    print("Resuming supplements:", len(done_supp), "papers inventoried")

for i, pmc in enumerate(paper_ids, 1):
    if pmc in done_supp:
        continue
    vdir = f"/papers/{pmc}/supplements/"
    listing = list_dir(vdir)
    # Keep only plausible filenames from ls output.
    files = [x.split()[-1] for x in listing if x and not x.endswith("/")]
    if not files:
        supp_rows.append({"paper_id":pmc,"filename":"","paperclip_path":vdir,
                          "local_path":"","bytes":0,"status":"none"})
    for fn in files:
        vpath = vdir + fn
        safe = re.sub(r"[^A-Za-z0-9._-]+","_",fn)
        local_dir = SUPP_DIR/pmc
        local_dir.mkdir(exist_ok=True)
        local = local_dir/safe
        try:
            if not local.exists():
                payload = paperclip_binary(vpath)
                if payload:
                    local.write_bytes(payload)
            supp_rows.append({
                "paper_id":pmc, "filename":fn, "paperclip_path":vpath,
                "local_path":str(local), "bytes":local.stat().st_size if local.exists() else 0,
                "status":"ok" if local.exists() else "empty"
            })
        except Exception as e:
            supp_rows.append({"paper_id":pmc,"filename":fn,"paperclip_path":vpath,
                              "local_path":str(local),"bytes":0,"status":repr(e)})
    if i % 20 == 0:
        pd.DataFrame(supp_rows).to_csv(supp_inv_path, sep="\t", index=False)
        print(i, "/", len(paper_ids))

supp_inv = pd.DataFrame(supp_rows)
supp_inv.to_csv(supp_inv_path, sep="\t", index=False)
print("Supplement entries:", len(supp_inv))
display(supp_inv.head())

## 7. Parse sequences from Paperclip-downloaded supplements

Supported directly: CSV, TSV, TXT, XLS/XLSX, JSON, XML and ZIP-contained tabular/text files.

The parser scans every textual cell, not just columns named `sequence`, because supplementary schemas are inconsistent.

In [ ]:
import io, zipfile
try:
    import openpyxl
except Exception:
    !pip -q install openpyxl
    import openpyxl

def iter_tables_from_file(path):
    """Yield (sheet_or_member, DataFrame) without any network access."""
    p = Path(path)
    ext = p.suffix.lower()
    try:
        if ext == ".csv":
            yield p.name, pd.read_csv(p, dtype=str, low_memory=False)
        elif ext in {".tsv",".tab"}:
            yield p.name, pd.read_csv(p, sep="\t", dtype=str, low_memory=False)
        elif ext in {".xls",".xlsx"}:
            book = pd.ExcelFile(p)
            for sh in book.sheet_names:
                yield sh, pd.read_excel(p, sheet_name=sh, dtype=str)
        elif ext in {".txt",".text",".lines"}:
            text = p.read_text(errors="replace")
            yield p.name, pd.DataFrame({"text": text.splitlines()})
        elif ext == ".json":
            obj = json.loads(p.read_text(errors="replace"))
            try:
                yield p.name, pd.json_normalize(obj)
            except Exception:
                yield p.name, pd.DataFrame({"text":[json.dumps(obj)]})
        elif ext == ".xml":
            text = p.read_text(errors="replace")
            yield p.name, pd.DataFrame({"text":[text]})
        elif ext == ".zip":
            with zipfile.ZipFile(p) as z:
                for member in z.namelist():
                    if member.endswith("/"): continue
                    data = z.read(member)
                    mex = Path(member).suffix.lower()
                    if mex == ".csv":
                        yield member, pd.read_csv(io.BytesIO(data), dtype=str, low_memory=False)
                    elif mex in {".tsv",".tab"}:
                        yield member, pd.read_csv(io.BytesIO(data), sep="\t", dtype=str, low_memory=False)
                    elif mex in {".txt",".text"}:
                        yield member, pd.DataFrame({"text":data.decode(errors="replace").splitlines()})
    except Exception as e:
        yield "__PARSE_ERROR__", pd.DataFrame({"text":[repr(e)]})

def seqs_from_text(s):
    if s is None or (isinstance(s,float) and np.isnan(s)): return []
    return sorted(set(SEQ_RE.findall(str(s).upper())))

supp_seq_rows = []
for _, r in supp_inv.iterrows():
    if r.get("status") != "ok" or not r.get("local_path"):
        continue
    local = r["local_path"]
    for sheet, df in iter_tables_from_file(local):
        # protect against gigantic tables while preserving all rows
        for col in df.columns:
            vals = df[col].astype(str)
            for ridx, val in vals.items():
                seqs = seqs_from_text(val)
                if not seqs:
                    continue
                # Include the complete row as compact evidence context.
                row_ctx = {str(k): str(v)[:500] for k,v in df.loc[ridx].to_dict().items()}
                ctx = json.dumps(row_ctx, ensure_ascii=False)[:5000]
                for seq in seqs:
                    supp_seq_rows.append({
                        "paper_id":r["paper_id"], "sequence":seq,
                        "origin":"supplement",
                        "paperclip_path":r["paperclip_path"],
                        "supplement_file":r["filename"],
                        "sheet_or_member":sheet,
                        "column":str(col),
                        "row_index":str(ridx),
                        "context":ctx
                    })

supp_seq = pd.DataFrame(supp_seq_rows)
if len(supp_seq):
    supp_seq = supp_seq.drop_duplicates(
        ["paper_id","sequence","paperclip_path","sheet_or_member","column","row_index"]
    )
supp_seq.to_csv(OUT/"supplement_sequence_candidates.tsv", sep="\t", index=False)
print("Supplement sequence candidates:", len(supp_seq))

## 8. Candidate cleanup + local evidence signals

A raw amino-acid-looking token can still be a false positive. We therefore:
- require 8–50 canonical residues;
- retain only papers already discovered via AMP/MIC terminology;
- score whether the same evidence context contains antimicrobial wet-lab terms;
- never set `wetlab=yes` merely because a peptide appears in a review.

The next section offers a stricter Paperclip AI-reader verification pass.

In [ ]:
inv = inventory.copy()
inv["paper_id"] = inv["paper_id"].astype(str)

parts = []
if len(body):
    b = body.copy()
    b["supplement_file"] = ""
    parts.append(b)
if len(supp_seq):
    s = supp_seq.copy()
    parts.append(s)

cand = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
if len(cand):
    cand = cand.merge(inv[["paper_id","doi","title","article_type","is_review"]],
                      on="paper_id", how="left")
    cand["sequence"] = cand["sequence"].str.upper()
    cand = cand[cand["sequence"].map(lambda x: MIN_LEN <= len(x) <= MAX_LEN and set(x) <= CANONICAL_AA)]
    wet_re = re.compile("|".join(re.escape(x.strip()) for x in WETLAB_TERMS if x.strip()), re.I)
    cand["context_wetlab_signal"] = cand["context"].fillna("").map(lambda x: bool(wet_re.search(x)))
    cand["is_review"] = cand["is_review"].astype(str).str.lower().isin(["true","1","yes"])
    cand["prelim_wetlab"] = cand["context_wetlab_signal"] & ~cand["is_review"]
    cand = cand.drop_duplicates(["paper_id","sequence","origin","paperclip_path","context"])

cand.to_csv(OUT/"amp_candidates.tsv", sep="\t", index=False)
print("Candidate rows:", len(cand))
print("Unique sequences:", cand["sequence"].nunique() if len(cand) else 0)
display(cand.head())

## 9. Paperclip AI-reader verification of primary papers

This is the **precision layer**.

For each primary paper containing candidate sequences, Paperclip `lookup pmc` creates a one-paper result set and `map` asks its AI reader to return strict JSON evidence.

For a very large full crawl this can be expensive, so:
- `VERIFY_MODE="pilot"` verifies the first 20 candidate papers;
- `VERIFY_MODE="all"` verifies every candidate primary paper;
- `VERIFY_MODE="off"` skips AI verification and leaves the master based on conservative deterministic evidence signals.

The prompt forbids inference and requires an **explicit sequence present in the paper/supplement**.

In [ ]:
VERIFY_MODE = "pilot"   # "off", "pilot", "all"
VERIFY_PILOT_N = 20

verify_out = OUT/"paperclip_verified_evidence.tsv"
verified_rows = []
verified_done = set()

if verify_out.exists():
    vv = pd.read_csv(verify_out, sep="\t", dtype=str).fillna("")
    verified_rows = vv.to_dict("records")
    verified_done = set(vv["paper_id"])
    print("Resuming verification:", len(verified_done), "papers done")

VERIFY_PROMPT = r"""
You are extracting antimicrobial-peptide evidence from ONE peer-reviewed paper.
Return ONLY a JSON array. Do not infer, guess, convert peptide names to sequences, or import facts from other papers.

For every EXPLICIT peptide amino-acid sequence in this paper that is presented as antimicrobial/antibacterial or experimentally tested against bacteria, return an object with:
{
 "sequence": "canonical sequence exactly as printed, uppercase if already canonical",
 "wetlab": "yes" or "no",
 "assay_type": ["MIC","MBC","time-kill","agar","CFU","hemolysis","HC50","cytotoxicity","in_vivo","other"],
 "organisms_or_strains": ["exact labels if reported"],
 "mic_values": ["verbatim values with units"],
 "hc50_or_toxicity": ["verbatim values with units/endpoints"],
 "in_vivo": "yes" or "no",
 "evidence_quote_or_location": "short identifying excerpt/table/section/location",
 "notes": "short factual description"
}

Rules:
- wetlab=yes ONLY if that exact sequence was experimentally tested in this paper.
- Computational prediction alone => wetlab=no.
- A sequence merely mentioned from prior literature => wetlab=no for this paper.
- Reviews should not create wetlab=yes evidence unless the review itself reports a new experiment.
- Preserve inequalities/ranges/units verbatim.
- If no explicit qualifying sequence exists, return [].
"""

if VERIFY_MODE != "off" and len(cand):
    primary_papers = sorted(cand.loc[~cand["is_review"], "paper_id"].dropna().unique())
    if VERIFY_MODE == "pilot":
        primary_papers = primary_papers[:VERIFY_PILOT_N]

    for i, pmc in enumerate(primary_papers, 1):
        if pmc in verified_done:
            continue
        print(f"[{i}/{len(primary_papers)}] verify {pmc}")
        try:
            lookup = pc_json(["lookup","pmc",pmc,"-n","1"], timeout=180)
            # result id can appear in JSON or formatted output
            txt = lookup if isinstance(lookup,str) else json.dumps(lookup)
            m = re.search(r"\bs_[A-Za-z0-9]+\b", txt)
            if not m:
                # Fallback: run non-JSON lookup to expose result id.
                txt2 = pc(["lookup","pmc",pmc,"-n","1"], timeout=180)
                m = re.search(r"\bs_[A-Za-z0-9]+\b", txt2)
            if not m:
                raise RuntimeError("No Paperclip search result ID from lookup")
            rid = m.group(0)

            mapped = pc(["map","--from",rid,VERIFY_PROMPT], timeout=600)
            # Recover the first JSON array in map output.
            jm = re.search(r"\[\s*\{.*\}\s*\]|\[\s*\]", mapped, flags=re.S)
            if not jm:
                verified_rows.append({"paper_id":pmc,"sequence":"","wetlab":"","notes":"",
                                      "raw_map":mapped[:10000],"status":"unparsed"})
            else:
                arr = json.loads(jm.group(0))
                for obj in arr:
                    verified_rows.append({
                        "paper_id":pmc,
                        "sequence":str(obj.get("sequence","")).replace(" ","").upper(),
                        "wetlab":obj.get("wetlab",""),
                        "assay_type":json.dumps(obj.get("assay_type",[]), ensure_ascii=False),
                        "organisms_or_strains":json.dumps(obj.get("organisms_or_strains",[]), ensure_ascii=False),
                        "mic_values":json.dumps(obj.get("mic_values",[]), ensure_ascii=False),
                        "hc50_or_toxicity":json.dumps(obj.get("hc50_or_toxicity",[]), ensure_ascii=False),
                        "in_vivo":obj.get("in_vivo",""),
                        "evidence_quote_or_location":obj.get("evidence_quote_or_location",""),
                        "notes":obj.get("notes",""),
                        "status":"ok"
                    })
        except Exception as e:
            verified_rows.append({"paper_id":pmc,"sequence":"","wetlab":"","notes":"",
                                  "status":"error","error":repr(e)})
        pd.DataFrame(verified_rows).to_csv(verify_out, sep="\t", index=False)

verified = pd.DataFrame(verified_rows)
if len(verified):
    verified.to_csv(verify_out, sep="\t", index=False)
print("Verified evidence rows:", len(verified))

## 10. Build evidence table and the requested four-column master TSV

Priority for `wetlab`:
1. Paperclip AI-reader verification if available.
2. Otherwise, conservative deterministic context evidence from a **primary** paper.

`source` is DOI when present, with PMC ID retained for auditability.

Multiple evidence records for the same sequence are aggregated into `notes`; `wetlab=yes` if at least one primary-paper evidence record qualifies.

In [ ]:
# Start from deterministic candidates.
ev = cand.copy() if len(cand) else pd.DataFrame()

if len(ev):
    ev["wetlab"] = ev["prelim_wetlab"].map({True:"yes",False:"no"})
    ev["assay_type"] = ""
    ev["organisms_or_strains"] = ""
    ev["mic_values"] = ""
    ev["hc50_or_toxicity"] = ""
    ev["in_vivo"] = ""
    ev["verified_by_map"] = False

# Overlay verified Paperclip map results by paper+sequence.
if len(verified) and len(ev):
    v = verified[(verified.get("status","")=="ok") & verified["sequence"].ne("")].copy()
    v = v[v["sequence"].map(lambda x: MIN_LEN <= len(x) <= MAX_LEN and set(x) <= CANONICAL_AA)]
    if len(v):
        # Add verified sequences not found by deterministic parser too.
        missing = v.merge(inv[["paper_id","doi","title","article_type","is_review"]],
                          on="paper_id", how="left")
        missing["origin"] = "paperclip_map"
        missing["paperclip_path"] = missing["paper_id"].map(lambda x:f"/papers/{x}/")
        missing["context"] = missing.get("evidence_quote_or_location","")
        missing["prelim_wetlab"] = missing["wetlab"].str.lower().eq("yes")
        missing["verified_by_map"] = True

        # For existing evidence rows, update exact paper+sequence matches.
        key_to_v = {(r.paper_id,r.sequence):r for _,r in v.iterrows()}
        for idx, r in ev.iterrows():
            k = (r["paper_id"], r["sequence"])
            if k in key_to_v:
                rr = key_to_v[k]
                ev.at[idx,"wetlab"] = str(rr.get("wetlab","")).lower()
                for col in ["assay_type","organisms_or_strains","mic_values","hc50_or_toxicity","in_vivo","notes"]:
                    if col in rr:
                        ev.at[idx,col] = rr.get(col,"")
                ev.at[idx,"verified_by_map"] = True

        # Append verified rows whose exact key isn't in ev.
        existing_keys = set(zip(ev["paper_id"],ev["sequence"]))
        add = missing[~missing.apply(lambda r:(r["paper_id"],r["sequence"]) in existing_keys,axis=1)]
        if len(add):
            ev = pd.concat([ev, add], ignore_index=True, sort=False)

# Clean impossible sequences once more.
if len(ev):
    ev["sequence"] = ev["sequence"].astype(str).str.upper().str.replace(r"\s+","",regex=True)
    ev = ev[ev["sequence"].map(lambda x: MIN_LEN <= len(x) <= MAX_LEN and set(x) <= CANONICAL_AA)]
    ev["source"] = ev.apply(
        lambda r: (str(r.get("doi","")).strip() + " | " if str(r.get("doi","")).strip() else "") + str(r["paper_id"]),
        axis=1
    )
    ev = ev.drop_duplicates(["paper_id","sequence","origin","paperclip_path"])
ev.to_csv(OUT/"amp_evidence.tsv", sep="\t", index=False)

# Aggregate to exactly the four requested columns.
master_rows = []
if len(ev):
    for seq, g in ev.groupby("sequence", sort=True):
        wet = "yes" if (g["wetlab"].astype(str).str.lower()=="yes").any() else "no"
        sources = sorted(set(x for x in g["source"].astype(str) if x))
        notes_bits = []
        for _,r in g.head(20).iterrows():  # compact master; full evidence remains in amp_evidence.tsv
            parts = [
                str(r.get("title","")).strip(),
                f"origin={r.get('origin','')}",
                f"MIC={r.get('mic_values','')}" if str(r.get("mic_values","")).strip() else "",
                f"tox={r.get('hc50_or_toxicity','')}" if str(r.get("hc50_or_toxicity","")).strip() else "",
                f"strains={r.get('organisms_or_strains','')}" if str(r.get("organisms_or_strains","")).strip() else "",
                str(r.get("notes","")).strip(),
            ]
            notes_bits.append("; ".join(x for x in parts if x))
        master_rows.append({
            "sequence":seq,
            "wetlab":wet,
            "source":" || ".join(sources),
            "notes":" || ".join(x for x in notes_bits if x)[:20000]
        })

master = pd.DataFrame(master_rows, columns=["sequence","wetlab","source","notes"])
stamp = datetime.utcnow().strftime("%Y%m%d")
master_path = OUT/f"amp_master_{stamp}.tsv"
master.to_csv(master_path, sep="\t", index=False, lineterminator="\n")

print("Evidence rows:", len(ev))
print("Unique AMP candidates:", len(master))
print("Wetlab=yes:", (master["wetlab"]=="yes").sum() if len(master) else 0)
print("MASTER:", master_path)
display(master.head(20))

## 11. Quality-control report

The master is intentionally only the four requested columns. This cell creates QA statistics without changing that schema.

In [ ]:
qa = {
    "generated_utc": datetime.utcnow().isoformat(),
    "mode": MODE,
    "candidate_papers": int(len(inventory)),
    "review_papers": int(inventory["is_review"].astype(str).str.lower().isin(["true","1","yes"]).sum()) if len(inventory) else 0,
    "body_candidate_rows": int(len(body)),
    "supplement_files": int((supp_inv["status"]=="ok").sum()) if len(supp_inv) else 0,
    "supplement_candidate_rows": int(len(supp_seq)),
    "evidence_rows": int(len(ev)),
    "unique_sequences": int(len(master)),
    "wetlab_yes": int((master["wetlab"]=="yes").sum()) if len(master) else 0,
    "wetlab_no": int((master["wetlab"]=="no").sum()) if len(master) else 0,
    "map_verified_rows": int(ev["verified_by_map"].astype(str).str.lower().eq("true").sum()) if len(ev) and "verified_by_map" in ev else 0,
}
(OUT/"qa_summary.json").write_text(json.dumps(qa, indent=2))
print(json.dumps(qa, indent=2))

if len(master):
    print("\nLength distribution:")
    print(master["sequence"].str.len().describe())
    print("\nRandom audit sample:")
    display(master.sample(min(20,len(master)), random_state=7))

## 12. Package outputs

Download the ZIP from Colab, or mount Google Drive and copy the output directory there.

Before committing to a final dataset release, run:
- `MODE="full"`
- `VERIFY_MODE="all"`

The most important audit file is `amp_evidence.tsv`; it preserves paper-level provenance beneath the convenient four-column master.

In [ ]:
archive = shutil.make_archive("/amp_pamperclip_results", "zip", OUT)
print("Created:", archive)
print("\nFiles:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUT), p.stat().st_size)